# Campus coffee sales: 7-day moving average

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Moving-Average Method** · a worked example from the StatNex teaching guide

**The story.** A campus coffee shop counted cups sold every day for four weeks, starting on a Monday. Weekdays are busy (175 to 223 cups) and weekends quiet (76 to 150), so the daily figures jump up and down every week and hide any slower change.

**The question.** Underneath the weekday-weekend swings, are sales growing?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/moving_average.csv` (28 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Day,Day of week,Cups sold
1.0,Monday,195.0
2.0,Tuesday,185.0
3.0,Wednesday,182.0
4.0,Thursday,191.0
5.0,Friday,188.0
6.0,Saturday,77.0
7.0,Sunday,76.0
8.0,Monday,187.0
9.0,Tuesday,197.0
10.0,Wednesday,175.0
11.0,Thursday,192.0
12.0,Friday,196.0
13.0,Saturday,112.0
14.0,Sunday,85.0
15.0,Monday,182.0
16.0,Tuesday,202.0
17.0,Wednesday,185.0
18.0,Thursday,207.0
19.0,Friday,194.0
20.0,Saturday,150.0
21.0,Sunday,87.0
22.0,Monday,204.0
23.0,Tuesday,199.0
24.0,Wednesday,202.0
25.0,Thursday,214.0
26.0,Friday,223.0
27.0,Saturday,139.0
28.0,Sunday,91.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# A time series in row order. The order of the rows is the data, so a series may not
# have blank cells: dropping one would make two non-adjacent periods neighbours.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Cups sold"]
# Each column is its own series.
series = {name: data[name] for name in columns}
span = 7
print(f"Span: {span}")

## Moving average

In [ ]:
# A centred moving average: each period gets the plain mean of the span periods around
# it, which pandas computes with rolling(span, center=True).mean(). When the span equals
# the length of a repeating cycle (7 for the days of the week), every window holds each
# phase of the cycle exactly once, so a cycle that repeats the same way cancels out and
# the trend is left. Read the column as the local level of the series. The first and
# last periods, where the window does not fit, stay empty (NaN): the newest periods
# have no trend value yet.
# An even span cannot be centred on a period; center=True puts the window half a step
# back (span // 2 periods before, span // 2 - 1 after), as StatNex does. Textbooks
# centre an even span with a "2 x span" moving average (an average of two neighbouring
# windows), which StatNex does not use.
tables = []
for name, values in series.items():
    values = values.astype(float).reset_index(drop=True)
    smoothed = values.rolling(span, center=True).mean()
    table = pd.DataFrame({"No.": np.arange(1, len(values) + 1), name: values,
                          "Moving average": smoothed})
    tables.append(table)
    print()
    print(table.to_string(index=False))

## What the result says

A moving average replaces each day's sales by the mean of the days around it. The span is the number of days in each mean. With a span of 7, every mean holds exactly one Monday, one Tuesday and so on, so the weekday and weekend swing cancels out.

The first moving average, centred on day 4, is 156.3 cups. The last, centred on day 25, is 181.7 cups. Between them the line climbs with only small steps back, while the daily counts jump between 76 and 223. The first 3 and last 3 days have no moving average, because a full week around them is missing.

So sales are growing underneath the weekly swing: the shop sells about 25 more cups a day in its last week than in its first.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.